# 06_compare_and_report (SmolLM-1.7B LoRA Pipeline)

This notebook is generated automatically to run on Kaggle.

### Goal & Metadata:
```
06_compare_and_report.ipynb
# Goal: consolidate all metrics, run qualitative evaluation,
```


In [ ]:
# Install required dependencies and resolve pre-installed torchao conflicts
!pip uninstall -y torchao
!pip install -q -U transformers trl datasets accelerate peft evaluate bitsandbytes huggingface_hub matplotlib pandas scikit-learn gradio


In [ ]:
%%writefile config.py
"""
config.py
Shared configuration + resume-state helpers for the SmolLM-1.7B LoRA pipeline.
Paste this as the FIRST cell in every notebook (01 through 06).
"""

import os
# Force single GPU to prevent PyTorch DataParallel device mismatch errors in Jupyter notebooks
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import random
import numpy as np
import torch

# ---------------------------------------------------------------------
# Paths (Kaggle-specific). /kaggle/working is ephemeral per-session but
# persists for the life of a session and is committed on "Save Version".
# ---------------------------------------------------------------------
BASE_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./work"
CKPT_DIR = os.path.join(BASE_DIR, "checkpoints")
METRICS_DIR = os.path.join(BASE_DIR, "metrics")
STATE_PATH = os.path.join(BASE_DIR, "state.json")

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

MODEL_ID = "HuggingFaceTB/SmolLM2-1.7B"          # upgraded to 1.7B parameter model
SFT_OUT_DIR = os.path.join(CKPT_DIR, "sft")
DPO_OUT_DIR = os.path.join(CKPT_DIR, "dpo")

# If you want checkpoints to survive session death, set these and log in
# with `huggingface_hub.login()` in the notebook before training.
PUSH_TO_HUB = False
HUB_SFT_REPO = "your-username/SmolLM-1.7B-SFT-LoRA"
HUB_DPO_REPO = "your-username/SmolLM-1.7B-DPO-LoRA"

SEED = 42

# ---------------------------------------------------------------------
# LoRA Parameters (PEFT)
# ---------------------------------------------------------------------
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
# Standard target modules for Llama/SmolLM architecture
LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# ---------------------------------------------------------------------
# ---------------------------------------------------------------------
# LLM Judge Configuration (Groq & Gemini fallbacks)
# ---------------------------------------------------------------------
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
if not GEMINI_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        GEMINI_API_KEY = user_secrets.get_secret("GEMINI_API_KEY")
    except Exception:
        pass

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
if not GROQ_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")
    except Exception:
        pass

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def is_main_process() -> bool:
    """True if running on the primary process under DDP, or if running single-process."""
    rank = int(os.environ.get("RANK", "0"))
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    return rank == 0 and local_rank == 0


def load_state() -> dict:
    if os.path.exists(STATE_PATH):
        with open(STATE_PATH) as f:
            return json.load(f)
    return {"phase": "none", "step": 0, "notes": ""}


def save_state(phase: str, step: int = 0, notes: str = ""):
    if not is_main_process():
        return
    state = {"phase": phase, "step": step, "notes": notes}
    with open(STATE_PATH, "w") as f:
        json.dump(state, f, indent=2)
    print(f"[state] phase={phase} step={step} notes={notes}")


def find_latest_checkpoint(output_dir: str):
    """Return path to latest `checkpoint-N` dir, or None."""
    if not os.path.isdir(output_dir):
        return None
    ckpts = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
    if not ckpts:
        return None
    ckpts.sort(key=lambda x: int(x.split("-")[-1]))
    return os.path.join(output_dir, ckpts[-1])


def save_metrics(name: str, metrics: dict):
    if not is_main_process():
        return
    path = os.path.join(METRICS_DIR, f"{name}_metrics.json")
    with open(path, "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"[metrics] saved -> {path}")
    print(json.dumps(metrics, indent=2))


def load_metrics(name: str) -> dict:
    path = os.path.join(METRICS_DIR, f"{name}_metrics.json")
    with open(path) as f:
        return json.load(f)


def push_checkpoint_to_dataset(local_dir: str, kaggle_dataset_slug: str):
    print(f"kaggle datasets version -p {local_dir} -m 'checkpoint update' -d {kaggle_dataset_slug}")


In [ ]:
%%writefile eval_utils.py
"""
eval_utils.py
One evaluation harness reused identically for base / SFT / DPO models so the
comparison in 06_compare_and_report is apples-to-apples. Supports PEFT loading and Gemini Judge.
"""

import os
import re
import json
import torch
import urllib.request
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def load_model_and_tokenizer(model_path: str, is_chat: bool = True):
    try:
        tok = AutoTokenizer.from_pretrained(model_path)
    except Exception:
        from config import MODEL_ID
        tok = AutoTokenizer.from_pretrained(MODEL_ID)
        
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    if tok.chat_template is None:
        tok.chat_template = (
            "{% for message in messages %}"
            "{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>\n'}}"
            "{% endfor %}"
            "{% if add_generation_prompt %}{{'<|im_start|>assistant\n'}}{% endif %}"
        )
        
    # Check if model_path is a PEFT/LoRA adapter (has adapter_config.json)
    is_peft = os.path.exists(os.path.join(model_path, "adapter_config.json"))
    
    if is_peft:
        # Load the base model first, then the PEFT wrapper
        from config import MODEL_ID, SFT_OUT_DIR, DPO_OUT_DIR
        from peft import PeftModel
        
        # If loading DPO adapter, the base model MUST be the merged SFT model
        if os.path.abspath(model_path) == os.path.abspath(DPO_OUT_DIR):
            print("Loading DPO adapter on top of SFT-merged base model...")
            raw_base = AutoModelForCausalLM.from_pretrained(
                MODEL_ID, torch_dtype=torch.float16
            ).to(DEVICE)
            sft_model = PeftModel.from_pretrained(raw_base, SFT_OUT_DIR).to(DEVICE)
            base_model = sft_model.merge_and_unload()
        else:
            # For SFT, the base is the raw base model
            base_model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID, torch_dtype=torch.float16
            ).to(DEVICE)
            
        model = PeftModel.from_pretrained(base_model, model_path).to(DEVICE)
    else:
        # Load standard model
        model = AutoModelForCausalLM.from_pretrained(
            model_path, torch_dtype=torch.float16
        ).to(DEVICE)
        
    model.eval()
    return model, tok


@torch.no_grad()
def generate(model, tok, prompt: str, chat: bool = True, max_new_tokens: int = 200):
    if chat and tok.chat_template is not None:
        messages = [{"role": "user", "content": prompt}]
        inputs = tok.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt"
        )
    else:
        inputs = tok(prompt, return_tensors="pt")

    # Extract input_ids tensor
    if isinstance(inputs, dict) or hasattr(inputs, "keys") or hasattr(inputs, "data"):
        input_ids = inputs["input_ids"]
    else:
        input_ids = inputs

    # Ensure input_ids is a torch.Tensor
    if not isinstance(input_ids, torch.Tensor):
        input_ids = torch.tensor(input_ids)

    input_ids = input_ids.to(DEVICE)

    # Ensure 2D tensor (batch_size, sequence_length)
    if input_ids.ndim == 1:
        input_ids = input_ids.unsqueeze(0)

    # Identify EOS tokens (standard eos + ChatML end token)
    eos_token_ids = [tok.eos_token_id]
    im_end_id = tok.convert_tokens_to_ids("<|im_end|>")
    if im_end_id is not None and im_end_id != tok.unk_token_id:
        eos_token_ids.append(im_end_id)

    out = model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=1.0,
        pad_token_id=tok.pad_token_id,
        eos_token_id=eos_token_ids,
    )
    text = tok.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
    return text.strip()


# ---------------------------------------------------------------------
# 1. Perplexity on a small held-out split (catastrophic-forgetting check)
# ---------------------------------------------------------------------
@torch.no_grad()
def compute_perplexity(model, tok, n_samples: int = 200, max_len: int = 512):
    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    ds = ds.filter(lambda x: len(x["text"].strip()) > 20).select(range(n_samples))

    nlls = []
    for idx, row in enumerate(ds):
        if idx > 0 and idx % 50 == 0:
            print(f"  [Perplexity] Processed {idx}/{n_samples} samples...")
        enc = tok(row["text"], return_tensors="pt", truncation=True, max_length=max_len)
        input_ids = enc.input_ids.to(DEVICE)
        if input_ids.shape[1] < 2:
            continue
        out = model(input_ids, labels=input_ids)
        nlls.append(out.loss.item())
    ppl = float(torch.exp(torch.tensor(nlls).mean()))
    return ppl


# ---------------------------------------------------------------------
# 2. Instruction-following
# ---------------------------------------------------------------------
INSTR_PROMPTS = [
    "List three benefits of regular exercise.",
    "Translate 'good morning' into French.",
    "Write a one-sentence summary of what photosynthesis is.",
    "Give me two synonyms for 'happy'.",
    "Explain what a variable is in programming, in one sentence.",
    "Name the capital of Japan.",
    "Convert 10 kilometers to miles.",
    "Write a short greeting for a birthday card.",
    "What is the boiling point of water in Celsius?",
    "Give a one-line definition of gravity.",
    # Multi-constraint and harder prompts
    "Answer the following question in exactly two sentences: What is the main cause of ocean tides?",
    "List exactly 4 items you would need to build a wooden birdhouse, no more and no less.",
    "Explain the concept of inflation to a 5-year-old in one paragraph of exactly three sentences.",
    "Respond ONLY with a JSON object containing the keys 'capital' and 'population' for the country France.",
    "Write a haiku about artificial intelligence.",
    "Give me a list of three distinct colors, but do not include the color blue or red.",
    "Tell me the name of the largest ocean on Earth, and write it in all capital letters.",
    "Provide two antonyms for the word 'cold', separated by a semicolon.",
    "In exactly five words, describe what a book is.",
    "What is the result of 15 multiplied by 4, plus 12? Provide only the final number.",
    "List three different programming languages that are statically typed.",
    "Write a short sentence where every word starts with the letter 'S'.",
    "Who wrote the play Romeo and Juliet? Answer with only the author's name.",
    "What are the three physical states of matter? List them separated by commas.",
    "State the name of the chemical symbol for Gold in exactly one word."
]


def score_instruction_following(model, tok, chat: bool = True):
    hits = 0
    outputs = []
    for idx, p in enumerate(INSTR_PROMPTS):
        resp = generate(model, tok, p, chat=chat, max_new_tokens=100)
        outputs.append({"prompt": p, "response": resp})
        
        # Check for repetition loops or ChatML collapses
        has_chat_collapse = any(marker in resp.lower() for marker in ["user\n", "assistant\n", "<|im_start|>", "<|im_end|>"])
        
        # Check for repeated lines of substantial length
        lines = [line.strip().lower() for line in resp.split("\n") if len(line.strip()) > 3]
        has_repetition = len(lines) != len(set(lines))
        
        resp_lower = resp.lower().strip()
        is_correct = False
        
        # Simple heuristic check for prompt-specific accuracy
        if "exercise" in p.lower():
            is_correct = any(w in resp_lower for w in ["benefit", "health", "weight", "heart", "cardio", "mental", "sleep", "energy"]) and len(resp_lower) > 20
        elif "french" in p.lower():
            is_correct = "bonjour" in resp_lower
        elif "photosynthesis" in p.lower():
            is_correct = "photosynthesis" in resp_lower or ("plant" in resp_lower and "light" in resp_lower)
        elif "synonyms for 'happy'" in p.lower():
            is_correct = any(w in resp_lower for w in ["joyful", "content", "cheerful", "glad", "delighted", "pleased", "merry", "elated"])
        elif "variable" in p.lower():
            is_correct = "variable" in resp_lower or "container" in resp_lower or "value" in resp_lower
        elif "japan" in p.lower():
            is_correct = "tokyo" in resp_lower
        elif "kilometers" in p.lower():
            is_correct = "6.2" in resp_lower or "6" in resp_lower
        elif "birthday" in p.lower():
            is_correct = "birthday" in resp_lower or "happy" in resp_lower or "greet" in resp_lower or "wish" in resp_lower
        elif "boiling point" in p.lower():
            is_correct = "100" in resp_lower
        elif "gravity" in p.lower():
            is_correct = "force" in resp_lower or "attract" in resp_lower or "gravity" in resp_lower
        elif "ocean tides" in p.lower():
            sentences = [s.strip() for s in re.split(r'[.!?]', resp.strip()) if s.strip()]
            is_correct = len(sentences) == 2 and any(w in resp_lower for w in ["moon", "gravit", "tide", "pull"])
        elif "birdhouse" in p.lower():
            is_correct = any(w in resp_lower for w in ["wood", "nail", "screw", "glue", "hammer", "saw", "paint"]) and ("1" in resp_lower or "2" in resp_lower or "3" in resp_lower or "4" in resp_lower or "-" in resp_lower or "\n" in resp_lower)
        elif "inflation" in p.lower():
            sentences = [s.strip() for s in re.split(r'[.!?]', resp.strip()) if s.strip()]
            is_correct = len(sentences) == 3 and any(w in resp_lower for w in ["price", "money", "buy", "rise", "cost"])
        elif "json" in p.lower():
            is_correct = "capital" in resp_lower and "population" in resp_lower and "paris" in resp_lower
        elif "haiku" in p.lower():
            lines_count = len([l for l in resp.split("\n") if l.strip()])
            is_correct = lines_count >= 2 and any(w in resp_lower for w in ["ai", "mind", "computer", "machine", "think", "code"])
        elif "distinct colors" in p.lower():
            is_correct = not any(w in resp_lower for w in ["blue", "red"]) and any(w in resp_lower for w in ["green", "yellow", "orange", "purple", "black", "white", "pink", "brown"])
        elif "largest ocean" in p.lower():
            is_correct = "PACIFIC" in resp
        elif "antonyms for the word 'cold'" in p.lower():
            is_correct = ";" in resp and any(w in resp_lower for w in ["hot", "warm", "heat"])
        elif "five words" in p.lower():
            words = [w for w in resp.split() if w.strip()]
            is_correct = len(words) == 5 and any(w in resp_lower for w in ["book", "read", "page", "story", "paper"])
        elif "15 multiplied by 4" in p.lower():
            is_correct = "72" in resp_lower
        elif "statically typed" in p.lower():
            is_correct = any(w in resp_lower for w in ["java", "c++", "c#", "rust", "go", "typescript", "swift", "kotlin", "scala"])
        elif "starts with the letter 's'" in p.lower():
            words = [w.strip(".,!?\"'") for w in resp_lower.split() if w.strip()]
            is_correct = len(words) >= 3 and all(w.startswith("s") for w in words)
        elif "romeo and juliet" in p.lower():
            is_correct = "shakespeare" in resp_lower
        elif "states of matter" in p.lower():
            is_correct = "solid" in resp_lower and "liquid" in resp_lower and "gas" in resp_lower
        elif "symbol for gold" in p.lower():
            is_correct = "au" in resp_lower
            
        ok = is_correct and not has_chat_collapse and not has_repetition
        if ok:
            hits += 1
    return {"accuracy_pct": 100 * hits / len(INSTR_PROMPTS), "samples": outputs}


# ---------------------------------------------------------------------
# 3. GSM8K subset: reasoning accuracy via exact-match on final number
# ---------------------------------------------------------------------
def extract_final_number(text: str):
    # Anchor to #### match first to prevent trailing text pollution
    ans_match = re.search(r"####\s*(-?\d[\d,]*\.?\d*)", text)
    if ans_match:
        return ans_match.group(1).replace(",", "")
    # Fallback to standard regex match for raw/chat completion outputs
    matches = re.findall(r"-?\d[\d,]*\.?\d*", text.replace(",", ""))
    return matches[-1] if matches else None


def score_gsm8k(model, tok, n_samples: int = 120, chat: bool = True):
    ds = load_dataset("openai/gsm8k", "main", split="test").select(range(n_samples))
    correct = 0
    cot_markers = 0
    outputs = []
    for idx, row in enumerate(ds):
        if idx > 0 and idx % 10 == 0:
            print(f"  [GSM8K] Solved {idx}/{n_samples} questions (Accuracy so far: {100 * correct / idx:.1f}%)...")
        q = row["question"]
        gold = row["answer"].split("####")[-1].strip().replace(",", "")
        resp = generate(
            model, tok,
            f"{q}\nSolve step by step and give the final numeric answer.",
            chat=chat, max_new_tokens=384,
        )
        pred = extract_final_number(resp)
        is_correct = pred is not None and pred == gold
        
        # Check for repetition loops
        lines = [line.strip().lower() for line in resp.split("\n") if len(line.strip()) > 3]
        has_repetition = len(lines) != len(set(lines))
        
        has_cot = bool(re.search(r"(step\s*\d|first,|then,|next,|\n\d\.)", resp, re.I)) and not has_repetition
        correct += int(is_correct)
        cot_markers += int(has_cot)
        outputs.append({"question": q, "gold": gold, "pred": pred, "response": resp})
    return {
        "accuracy_pct": 100 * correct / n_samples,
        "cot_presence_pct": 100 * cot_markers / n_samples,
        "samples": outputs[:10],
    }


# ---------------------------------------------------------------------
# 4. Full eval bundle for one checkpoint
# ---------------------------------------------------------------------
def run_full_eval(model_path: str, chat: bool = True, ppl_n=200, instr_n=None, gsm8k_n=120):
    model, tok = load_model_and_tokenizer(model_path, is_chat=chat)
    print(f"Computing perplexity on {ppl_n} wikitext-2 samples...")
    ppl = compute_perplexity(model, tok, n_samples=ppl_n)
    print(f"Scoring instruction following ({len(INSTR_PROMPTS)} prompts)...")
    instr = score_instruction_following(model, tok, chat=chat)
    print(f"Scoring GSM8K reasoning on {gsm8k_n} problems...")
    gsm8k = score_gsm8k(model, tok, n_samples=gsm8k_n, chat=chat)

    del model
    torch.cuda.empty_cache()

    return {
        "perplexity": ppl,
        "instruction_following_acc_pct": instr["accuracy_pct"],
        "gsm8k_acc_pct": gsm8k["accuracy_pct"],
        "cot_presence_pct": gsm8k["cot_presence_pct"],
        "instr_samples": instr["samples"],
        "gsm8k_samples": gsm8k["samples"],
    }


# ---------------------------------------------------------------------
# 5. Gemini API Judge (LLM-as-a-Judge)
# ---------------------------------------------------------------------
def get_gemini_judge_score(prompt: str, base_resp: str, sft_resp: str, dpo_resp: str) -> dict:
    from config import GEMINI_API_KEY, GROQ_API_KEY
    if not GEMINI_API_KEY and not GROQ_API_KEY:
        return {"winner": "Tie", "reason": "No Gemini or Groq API key configured."}
    
    judge_prompt = f"""
    You are an expert AI evaluator. Compare three responses generated by different models (Base, SFT, DPO) to the user's prompt.
    Evaluate their quality based on helpfulness, accuracy, structure, instruction following, and step-by-step reasoning quality.
    
    [USER PROMPT]:
    {prompt}
    
    [BASE RESPONSE]:
    {base_resp}
    
    [SFT RESPONSE]:
    {sft_resp}
    
    [DPO RESPONSE]:
    {dpo_resp}
    
    Respond ONLY with a JSON object in this format:
    {{
      "winner": "Base" | "SFT" | "DPO" | "Tie",
      "reason": "<one sentence explanation of why the winner was chosen>"
    }}
    """
    
    # 1. Try Groq first if available
    if GROQ_API_KEY:
        url = "https://api.groq.com/openai/v1/chat/completions"
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {GROQ_API_KEY}",
            "User-Agent": "Mozilla/5.0"
        }
        models_to_try = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant", "llama3-70b-8192", "llama3-8b-8192", "mixtral-8x7b-32768"]
        for groq_model in models_to_try:
            data = {
                "model": groq_model,
                "messages": [
                    {"role": "user", "content": judge_prompt}
                ],
                "response_format": {"type": "json_object"}
            }
            try:
                req = urllib.request.Request(
                    url,
                    data=json.dumps(data).encode("utf-8"),
                    headers=headers
                )
                with urllib.request.urlopen(req, timeout=15) as response:
                    res = json.loads(response.read().decode("utf-8"))
                    text = res["choices"][0]["message"]["content"]
                    return json.loads(text.strip())
            except Exception:
                continue
            
    # 2. Try Gemini as fallback if available
    if GEMINI_API_KEY:
        gemini_models = ["gemini-3.5-flash", "gemini-3.5-flash-lite", "gemini-2.5-flash", "gemini-1.5-flash"]
        for gem_m in gemini_models:
            url = f"https://generativelanguage.googleapis.com/v1beta/models/{gem_m}:generateContent?key={GEMINI_API_KEY}"
            data = {
                "contents": [{
                    "parts": [{"text": judge_prompt}]
                }],
                "generationConfig": {
                    "responseMimeType": "application/json"
                }
            }
            try:
                req = urllib.request.Request(
                    url,
                    data=json.dumps(data).encode("utf-8"),
                    headers={"Content-Type": "application/json", "User-Agent": "Mozilla/5.0"}
                )
                with urllib.request.urlopen(req, timeout=15) as response:
                    res = json.loads(response.read().decode("utf-8"))
                    text = res["candidates"][0]["content"]["parts"][0]["text"]
                    return json.loads(text.strip())
            except Exception:
                continue
            
    return {"winner": "Error", "reason": "All API judge model calls failed or timed out."}


# ---------------------------------------------------------------------
# 6. Offline DPO Metrics Calculator (Option 2)
# ---------------------------------------------------------------------
@torch.no_grad()
def compute_model_logps_single(model, tok, prompt_msgs, completion_msgs):
    # Retrieve token ids for the prompt
    prompt_res = tok.apply_chat_template(prompt_msgs, add_generation_prompt=True)
    if isinstance(prompt_res, dict) or hasattr(prompt_res, "input_ids") or hasattr(prompt_res, "data"):
        prompt_ids = prompt_res["input_ids"]
    else:
        prompt_ids = prompt_res
        
    # Convert prompt_ids to a plain python list of integers
    if hasattr(prompt_ids, "tolist"):
        prompt_ids = prompt_ids.tolist()
    else:
        prompt_ids = list(prompt_ids)
        
    if len(prompt_ids) > 0 and isinstance(prompt_ids[0], list):
        prompt_ids = prompt_ids[0]
    
    # Retrieve completion text and tokenize it directly
    completion_text = completion_msgs[0]["content"]
    comp_res = tok.encode(completion_text, add_special_tokens=False)
    if isinstance(comp_res, dict) or hasattr(comp_res, "input_ids") or hasattr(comp_res, "data"):
        completion_ids = comp_res["input_ids"]
    else:
        completion_ids = comp_res
        
    # Convert completion_ids to a plain python list of integers
    if hasattr(completion_ids, "tolist"):
        completion_ids = completion_ids.tolist()
    else:
        completion_ids = list(completion_ids)
        
    if len(completion_ids) > 0 and isinstance(completion_ids[0], list):
        completion_ids = completion_ids[0]
    
    # Concatenate prompt + completion
    full_ids = prompt_ids + completion_ids
    
    # Ensure standard termination tag (im_end or eos) is present at the end
    eos_id = tok.eos_token_id
    im_end_id = tok.convert_tokens_to_ids("<|im_end|>")
    if im_end_id is not None and im_end_id != tok.unk_token_id:
        if not full_ids or full_ids[-1] != im_end_id:
            full_ids.append(im_end_id)
    else:
        if not full_ids or full_ids[-1] != eos_id:
            full_ids.append(eos_id)
            
    # Convert list to PyTorch tensor
    full_ids_tensor = torch.tensor(full_ids)
    
    # Labels: set prompt tokens to -100 (ignored in loss computation)
    labels = full_ids_tensor.clone()
    labels[:len(prompt_ids)] = -100
    
    # Input batch formatting
    input_ids = full_ids_tensor.unsqueeze(0).to(DEVICE)
    labels = labels.unsqueeze(0).to(DEVICE)
    
    outputs = model(input_ids)
    logits = outputs.logits
    
    # Compute log probabilities
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    
    loss_mask = shift_labels != -100
    shift_labels[shift_labels == -100] = 0
    
    log_probs = shift_logits.log_softmax(-1)
    per_token_logps = torch.gather(log_probs, dim=2, index=shift_labels.unsqueeze(-1)).squeeze(-1)
    
    # Zero out ignored prompt tokens
    per_token_logps = per_token_logps * loss_mask
    return per_token_logps.sum(-1).item()


def compute_dataset_logps(model_path, dataset_samples, is_chat=True):
    model, tok = load_model_and_tokenizer(model_path, is_chat=is_chat)
    chosen_logps = []
    rejected_logps = []
    
    for idx, ex in enumerate(dataset_samples):
        if idx > 0 and idx % 20 == 0:
            print(f"  [Logps] Computed {idx}/{len(dataset_samples)} samples...")
        
        # Handle prompt
        if "prompt" in ex:
            prompt_val = ex["prompt"]
            if isinstance(prompt_val, list):
                prompt_msgs = prompt_val
            else:
                prompt_msgs = [{"role": "user", "content": str(prompt_val)}]
        else:
            system_prompt = ex.get("system", "")
            user_prompt = ex.get("input", "") or ex.get("question", "")
            prompt_msgs = []
            if system_prompt:
                prompt_msgs.append({"role": "system", "content": system_prompt})
            prompt_msgs.append({"role": "user", "content": user_prompt})
            
        # Handle chosen and rejected
        chosen_val = ex["chosen"]
        if isinstance(chosen_val, list):
            chosen_content = chosen_val[-1]["content"] if (len(chosen_val) > 0 and isinstance(chosen_val[-1], dict)) else str(chosen_val)
        else:
            chosen_content = str(chosen_val)
            
        rejected_val = ex["rejected"]
        if isinstance(rejected_val, list):
            rejected_content = rejected_val[-1]["content"] if (len(rejected_val) > 0 and isinstance(rejected_val[-1], dict)) else str(rejected_val)
        else:
            rejected_content = str(rejected_val)
            
        chosen_msgs = [{"role": "assistant", "content": chosen_content}]
        rejected_msgs = [{"role": "assistant", "content": rejected_content}]
        
        try:
            chosen_lp = compute_model_logps_single(model, tok, prompt_msgs, chosen_msgs)
            rejected_lp = compute_model_logps_single(model, tok, prompt_msgs, rejected_msgs)
            chosen_logps.append(chosen_lp)
            rejected_logps.append(rejected_lp)
        except Exception as e:
            # Handle template errors safely
            print(f"Error computing log-prob for index {idx}: {e}")
            chosen_logps.append(0.0)
            rejected_logps.append(0.0)
            
    del model
    torch.cuda.empty_cache()
    return chosen_logps, rejected_logps


In [ ]:
# and query the Gemini API Judge to compute model Win Rates.
# Runtime: ~3-5 min
# =====================================================================


if __name__ == "__main__":
    import os
    from config import load_metrics, save_state, BASE_DIR
    import pandas as pd
    import matplotlib.pyplot as plt
    
    save_state(phase="report_start")
    
    baseline = load_metrics("baseline")
    sft = load_metrics("sft")
    dpo = load_metrics("dpo")
    
    def pct_identical(samples_a, samples_b):
        if not samples_a or not samples_b or len(samples_a) != len(samples_b):
            return 0.0
        matches = sum(a["response"] == b["response"] for a, b in zip(samples_a, samples_b))
        return 100.0 * matches / len(samples_a)
        
    gsm8k_base_sft_overlap = pct_identical(baseline.get("gsm8k_samples", []), sft.get("gsm8k_samples", []))
    gsm8k_sft_dpo_overlap = pct_identical(sft.get("gsm8k_samples", []), dpo.get("gsm8k_samples", []))
    instr_base_sft_overlap = pct_identical(baseline.get("instr_samples", []), sft.get("instr_samples", []))
    instr_sft_dpo_overlap = pct_identical(sft.get("instr_samples", []), dpo.get("instr_samples", []))
    
    df = pd.DataFrame({
        "Metric": ["Perplexity", "Instruction-Follow Acc (%)", "GSM8K Acc (%)", "CoT Presence (%)"],
        "Base": [baseline["perplexity"], baseline["instruction_following_acc_pct"],
                 baseline["gsm8k_acc_pct"], baseline["cot_presence_pct"]],
        "SFT": [sft["perplexity"], sft["instruction_following_acc_pct"],
                sft["gsm8k_acc_pct"], sft["cot_presence_pct"]],
        "DPO": [dpo["perplexity"], dpo["instruction_following_acc_pct"],
                dpo["gsm8k_acc_pct"], dpo["cot_presence_pct"]],
    })
    print("\n--- Quantitative Summary ---")
    print(df.to_string(index=False))
    print(f"\n--- Output Overlap (Identity Checks) ---")
    print(f"Base vs SFT GSM8K Overlap:       {gsm8k_base_sft_overlap:.1f}%")
    print(f"SFT vs DPO GSM8K Overlap:        {gsm8k_sft_dpo_overlap:.1f}%")
    print(f"Base vs SFT Instruction Overlap: {instr_base_sft_overlap:.1f}%")
    print(f"SFT vs DPO Instruction Overlap:  {instr_sft_dpo_overlap:.1f}%")
    df.to_csv(os.path.join(BASE_DIR, "final_comparison.csv"), index=False)
    
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    metrics = ["Instruction-Follow Acc (%)", "GSM8K Acc (%)", "CoT Presence (%)", "Perplexity"]
    for ax, metric in zip(axes, metrics):
        row = df[df["Metric"] == metric].iloc[0]
        ax.bar(["Base", "SFT", "DPO"], [row["Base"], row["SFT"], row["DPO"]],
               color=["#94a3b8", "#60a5fa", "#34d399"])
        ax.set_title(metric, fontsize=10)
        ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, "progression_chart.png"), dpi=150)
    plt.close()
    
    ### CELL 3: generate outputs for qualitative showcase ###
    from eval_utils import load_model_and_tokenizer, generate, get_gemini_judge_score
    from config import MODEL_ID, SFT_OUT_DIR, DPO_OUT_DIR, GEMINI_API_KEY, GROQ_API_KEY
    import torch
    
    showcase_prompts = [
        "If a train travels 60 miles in 1.5 hours, what is its average speed?",
        "Explain in simple terms why the sky is blue.",
        "A store has 120 apples. They sell 45 in the morning and 30 in the afternoon. How many are left?",
        "What are three tips for staying focused while studying?",
        "If I have $50 and spend 40% of it, how much do I have left?",
        "Write a polite email to a customer explaining that their shipment will be delayed by 2 days due to weather conditions.",
        "What is the difference between a software library and a framework? Explain with a simple analogy."
    ]
    
    responses = {p: {} for p in showcase_prompts}
    
    for label, path, chat in [
        ("Base", MODEL_ID, False),
        ("SFT", SFT_OUT_DIR, True),
        ("DPO", DPO_OUT_DIR, True),
    ]:
        print(f"Generating answers from {label} model...")
        model, tok = load_model_and_tokenizer(path, is_chat=chat)
        for p in showcase_prompts:
            resp = generate(model, tok, p, chat=chat, max_new_tokens=250)
            responses[p][label] = resp
        del model
        torch.cuda.empty_cache()
    
    ### CELL 4: LLM-as-a-Judge Evaluation (Win Rate) ###
    judge_results = []
    win_counts = {"Base": 0, "SFT": 0, "DPO": 0, "Tie": 0, "Error": 0}
    
    if GROQ_API_KEY or GEMINI_API_KEY:
        if GROQ_API_KEY:
            print("\nQuerying Groq API Judge (LLM-as-a-Judge)...")
        else:
            print("\nQuerying Gemini API Judge (LLM-as-a-Judge)...")
        for idx, prompt in enumerate(showcase_prompts):
            print(f"  Evaluating Prompt {idx+1}/{len(showcase_prompts)}...")
            base_ans = responses[prompt]["Base"]
            sft_ans = responses[prompt]["SFT"]
            dpo_ans = responses[prompt]["DPO"]
            
            judge_opinion = get_gemini_judge_score(prompt, base_ans, sft_ans, dpo_ans)
            winner = judge_opinion.get("winner", "Tie")
            reason = judge_opinion.get("reason", "No reason provided.")
            
            win_counts[winner] = win_counts.get(winner, 0) + 1
            judge_results.append({
                "prompt": prompt,
                "Base": base_ans,
                "SFT": sft_ans,
                "DPO": dpo_ans,
                "winner": winner,
                "reason": reason
            })
            
        judge_df = pd.DataFrame(judge_results)
        judge_df.to_csv(os.path.join(BASE_DIR, "llm_judge_results.csv"), index=False)
        
        print("\n--- LLM-as-a-Judge Win Rates ---")
        total_valid = sum(win_counts[k] for k in ["Base", "SFT", "DPO", "Tie"])
        if total_valid > 0:
            for model_name in ["Base", "SFT", "DPO", "Tie"]:
                pct = 100 * win_counts[model_name] / total_valid
                print(f"Model {model_name:<5} | Wins: {win_counts[model_name]} | Win Rate: {pct:.1f}%")
        
        # Display side-by-side results
        for item in judge_results:
            print(f"\n{'='*80}\nPROMPT: {item['prompt']}\n{'='*80}")
            print(f"[Base]: {item['Base']}\n")
            print(f"[SFT]: {item['SFT']}\n")
            print(f"[DPO]: {item['DPO']}\n")
            print(f"👉 JUDGE WINNER: {item['winner']}")
            print(f"👉 REASON: {item['reason']}")
    else:
        print("\n[Notice] No active API keys (Groq/Gemini) found. Skipping LLM-as-a-Judge evaluation.")
        # Just save regular outputs without judge evaluation
        showcase_rows = []
        for prompt, resps in responses.items():
            showcase_rows.append({
                "prompt": prompt,
                "Base": resps["Base"],
                "SFT": resps["SFT"],
                "DPO": resps["DPO"]
            })
        pd.DataFrame(showcase_rows).to_csv(os.path.join(BASE_DIR, "qualitative_showcase.csv"), index=False)
        for prompt in showcase_prompts:
            print(f"\n{'='*80}\nPROMPT: {prompt}\n{'='*80}")
            for label in ["Base", "SFT", "DPO"]:
                print(f"\n[{label}]\n{responses[prompt][label]}")
                
    # ---------------------------------------------------------------------
    # OPTION 2: Compute Offline Validation DPO Margin and Accuracy
    # ---------------------------------------------------------------------
    print("\n--- Running Offline Validation DPO Margin & Accuracy ---")
    try:
        from datasets import load_dataset
        from eval_utils import compute_dataset_logps
        import numpy as np
        import json
        
        print("Loading HuggingFaceH4/ultrafeedback_binarized (test_prefs)...")
        raw_eval = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="test_prefs")
        
        MAX_CHARS = 1200
        def is_high_quality_and_short(ex):
            chosen = ex["chosen"]
            rejected = ex["rejected"]
            chosen_text = chosen[-1]["content"] if isinstance(chosen, list) else str(chosen)
            rejected_text = rejected[-1]["content"] if isinstance(rejected, list) else str(rejected)
            if len(chosen_text) > MAX_CHARS or len(rejected_text) > MAX_CHARS:
                return False
            score_c = ex.get("score_chosen", 0.0)
            score_r = ex.get("score_rejected", 0.0)
            if score_c is not None and score_r is not None:
                if (score_c - score_r) < 1.0:
                    return False
            return True
            
        filtered_eval = raw_eval.filter(is_high_quality_and_short, num_proc=4).shuffle(seed=42)
        dpo_eval_samples = filtered_eval.select(range(min(100, len(filtered_eval))))
        
        print("Computing log probabilities under SFT Model...")
        sft_chosen_lps, sft_rejected_lps = compute_dataset_logps(SFT_OUT_DIR, dpo_eval_samples)
        
        print("Computing log probabilities under DPO Model...")
        dpo_chosen_lps, dpo_rejected_lps = compute_dataset_logps(DPO_OUT_DIR, dpo_eval_samples)
        
        # Calculate rewards and margins
        beta = 0.05
        sft_chosen_lps = np.array(sft_chosen_lps)
        sft_rejected_lps = np.array(sft_rejected_lps)
        dpo_chosen_lps = np.array(dpo_chosen_lps)
        dpo_rejected_lps = np.array(dpo_rejected_lps)
        
        dpo_chosen_rewards = beta * (dpo_chosen_lps - sft_chosen_lps)
        dpo_rejected_rewards = beta * (dpo_rejected_lps - sft_rejected_lps)
        
        margins = dpo_chosen_rewards - dpo_rejected_rewards
        dpo_acc = float((margins > 0).mean()) * 100.0
        avg_margin = float(margins.mean())
        
        print("\n==================================================")
        print("📊 OFFLINE DPO VALIDATION RESULTS (Orca Domain)")
        print("==================================================")
        print(f"Validation Preference Accuracy: {dpo_acc:.2f}%")
        print(f"Average Reward Margin:           {avg_margin:.6f}")
        print("==================================================")
        
        # Save metrics
        dpo_val_metrics = {
            "validation_preference_accuracy": dpo_acc,
            "average_reward_margin": avg_margin
        }
        with open(os.path.join(BASE_DIR, "dpo_validation_analysis.json"), "w") as f:
            json.dump(dpo_val_metrics, f, indent=2)
            
    except Exception as e:
        print(f"Skipping or failed DPO Validation Analysis: {e}")
        
    save_state(phase="report_done", notes="progression_chart.png, final_comparison.csv, outputs saved.")
    print("\nReport generation complete.")
